Code created by Faith Townsend to access NOAA ROMS model data (provided by Alexander Kurapov at NOAA), pull it from 
.nc files into csvs that can be more easily processed to do the analyses presented in Hamilton et al 2027.

In [3]:
#Import neccessary packages
import os
import ftplib
import xarray as xr
import pandas as pd

In [ ]:
# Define FTP details
FTP_HOST = "ocsftp.ncd.noaa.gov"
FTP_DIR = "/Khazaei/4OregonKelp/TSzeta/"
LOCAL_DIR = "nc_files"

In [ ]:
# Create a local directory if it doesn't exist
os.makedirs(LOCAL_DIR, exist_ok=True)

In [ ]:
# Connect to FTP and list .nc files
ftp = ftplib.FTP(FTP_HOST)
ftp.login()  # Assumes anonymous login
ftp.cwd(FTP_DIR)

In [ ]:
# Get list of .nc files
nc_files = [f for f in ftp.nlst() if f.endswith(".nc")]

In [ ]:
# Download .nc files in batches of 500
batch_size = 500

for i in range(0, len(nc_files), batch_size):
    batch = nc_files[i:i + batch_size]  # Get a batch of 500 files
    for file in batch:
        local_path = os.path.join(LOCAL_DIR, file)
        with open(local_path, "wb") as f:
            ftp.retrbinary(f"RETR {file}", f.write)
        print(f"Downloaded: {file}")
    
    print(f"Batch {i // batch_size + 1} completed.")

ftp.quit()

In [ ]:
print(ds.s_rho.values)  # Check order

In [25]:
data_dir = r"D:/Oceanographic_Kelp_Project/nc_files"
output_dir = r"D:/Oceanographic_Kelp_Project/nc_files_adj"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

# Get all NetCDF files in the directory
nc_files = [f for f in os.listdir(data_dir) if f.endswith(".nc")]

for file in nc_files:
    file_path = os.path.join(data_dir, file)
    ds = xr.open_dataset(file_path)
    
    # Ensure ocean_time is in datetime format
    ds['ocean_time'] = pd.to_datetime(ds['ocean_time'].values)
    
    # Check required variables exist
    required_vars = ['salt', 'temp', 'lon_rho', 'lat_rho']
    if not all(var in ds.variables for var in required_vars):
        print(f"Skipping {file}, missing required variables.")
        continue
    
    # Extract surface salinity and temperature
    surf_salt = ds.salt[:, -1, :, :]
    surf_temp = ds.temp[:, -1, :, :]
    
    for date in ds.ocean_time.values:  # Directly iterate over available ocean_time values
        daily_ds = ds.sel(ocean_time=date)
        
        # Create daily datasets
        salt_ds = xr.Dataset({
            "ocean_time": daily_ds.ocean_time,
            "lat_rho": daily_ds.lat_rho,
            "lon_rho": daily_ds.lon_rho,
            "surf_salt": surf_salt.sel(ocean_time=date)
        })
        
        temp_ds = xr.Dataset({
            "ocean_time": daily_ds.ocean_time,
            "lat_rho": daily_ds.lat_rho,
            "lon_rho": daily_ds.lon_rho,
            "surf_temp": surf_temp.sel(ocean_time=date)
        })
        
        # Save daily files
        salt_filename = os.path.join(output_dir, f"salt_{pd.Timestamp(date).date()}.nc")
        temp_filename = os.path.join(output_dir, f"temp_{pd.Timestamp(date).date()}.nc")
        
        salt_ds.to_netcdf(salt_filename)
        temp_ds.to_netcdf(temp_filename)
        
        print(f"Created: {salt_filename}")
        print(f"Created: {temp_filename}")
        
        del salt_ds, temp_ds  # Free memory
    
    ds.close()

Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-01.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-01.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-02.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-02.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-03.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-03.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-04.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-04.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-05.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-05.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-06.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-06.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\salt_2008-10-07.nc
Created: D:/Oceanographic_Kelp_Project/nc_files_adj\temp_2008-10-07.nc
Create

In [31]:
data_dir = r"D:/Oceanographic_Kelp_Project/nc_files_adj"
output_dir = r"D:/Oceanographic_Kelp_Project/nc_files_adj"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

# Combine salt and temp files by year
salt_files = [f for f in os.listdir(output_dir) if f.startswith("salt_")]
temp_files = [f for f in os.listdir(output_dir) if f.startswith("temp_")]

years = range(2008, 2019)

for year in years:
    yearly_salt_files = [f for f in salt_files if f"salt_{year}" in f]
    yearly_temp_files = [f for f in temp_files if f"temp_{year}" in f]
    
    print(f"Loading salt files for year {year}")
    salt_datasets = [xr.open_dataset(os.path.join(output_dir, f)) for f in yearly_salt_files]
    
    print(f"Loading temp files for year {year}")
    temp_datasets = [xr.open_dataset(os.path.join(output_dir, f)) for f in yearly_temp_files]
    
    if salt_datasets:
        surf_salt_yearly = xr.concat(salt_datasets, dim="ocean_time")
        surf_salt_yearly.to_netcdf(os.path.join(output_dir, f"surf_salt_{year}.nc"))
        print(f"Created: surf_salt_{year}.nc")
    
    if temp_datasets:
        surf_temp_yearly = xr.concat(temp_datasets, dim="ocean_time")
        surf_temp_yearly.to_netcdf(os.path.join(output_dir, f"surf_temp_{year}.nc"))
        print(f"Created: surf_temp_{year}.nc")
    
    # Close datasets
    for ds in salt_datasets + temp_datasets:
        ds.close()


Loading salt files for year 2008
Loading temp files for year 2008
Created: surf_salt_2008.nc
Created: surf_temp_2008.nc
Loading salt files for year 2009
Loading temp files for year 2009
Created: surf_salt_2009.nc
Created: surf_temp_2009.nc
Loading salt files for year 2010
Loading temp files for year 2010
Created: surf_salt_2010.nc
Created: surf_temp_2010.nc
Loading salt files for year 2011
Loading temp files for year 2011
Created: surf_salt_2011.nc
Created: surf_temp_2011.nc
Loading salt files for year 2012
Loading temp files for year 2012
Created: surf_salt_2012.nc
Created: surf_temp_2012.nc
Loading salt files for year 2013
Loading temp files for year 2013
Created: surf_salt_2013.nc
Created: surf_temp_2013.nc
Loading salt files for year 2014
Loading temp files for year 2014
Created: surf_salt_2014.nc
Created: surf_temp_2014.nc
Loading salt files for year 2015
Loading temp files for year 2015
Created: surf_salt_2015.nc
Created: surf_temp_2015.nc
Loading salt files for year 2016
Loading

In [43]:
# Set the file directory for the processed files
data_dir = r"D:/Oceanographic_Kelp_Project/nc_files_adj"
output_dir = r"D:/Oceanographic_Kelp_Project/NOAA_all"
os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

# List all NetCDF files for salt and temperature data
salt_files = [f for f in os.listdir(data_dir) if f.startswith("surf_salt_")]
temp_files = [f for f in os.listdir(data_dir) if f.startswith("surf_temp_")]

# Process each year's data
years = range(2008, 2019)

for year in years:
    # Get the salt and temp files for the specific year
    yearly_salt_file = next((f for f in salt_files if f"surf_salt_{year}" in f), None)
    yearly_temp_file = next((f for f in temp_files if f"surf_temp_{year}" in f), None)

    if yearly_salt_file and yearly_temp_file:
        # Load the datasets for the specific year
        print(f"Loading file for year {year}")
        
        salt_ds = xr.open_dataset(os.path.join(data_dir, yearly_salt_file))
        temp_ds = xr.open_dataset(os.path.join(data_dir, yearly_temp_file))

        # Convert salt and temp datasets to DataFrames
        salt_df = salt_ds.to_dataframe().reset_index()
        temp_df = temp_ds.to_dataframe().reset_index()

        # Export each DataFrame as a separate CSV file
        salt_df.to_csv(os.path.join(output_dir, f"surf_salt_{year}.csv"), index=False)
        print(f"Exported salt data for {year} as surf_salt_{year}.csv")

        temp_df.to_csv(os.path.join(output_dir, f"surf_temp_{year}.csv"), index=False)
        print(f"Exported temperature data for {year} as surf_temp_{year}.csv")

        # Close the datasets
        salt_ds.close()
        temp_ds.close()
    else:
        print(f"Missing files for year {year}, skipping.")


Loading file for year 2008
Exported salt data for 2008 as surf_salt_2008.csv
Exported temperature data for 2008 as surf_temp_2008.csv
Loading file for year 2009
Exported salt data for 2009 as surf_salt_2009.csv
Exported temperature data for 2009 as surf_temp_2009.csv
Loading file for year 2010
Exported salt data for 2010 as surf_salt_2010.csv
Exported temperature data for 2010 as surf_temp_2010.csv
Loading file for year 2011
Exported salt data for 2011 as surf_salt_2011.csv
Exported temperature data for 2011 as surf_temp_2011.csv
Loading file for year 2012
Exported salt data for 2012 as surf_salt_2012.csv
Exported temperature data for 2012 as surf_temp_2012.csv
Loading file for year 2013
Exported salt data for 2013 as surf_salt_2013.csv
Exported temperature data for 2013 as surf_temp_2013.csv
Loading file for year 2014
Exported salt data for 2014 as surf_salt_2014.csv
Exported temperature data for 2014 as surf_temp_2014.csv
Loading file for year 2015
Exported salt data for 2015 as surf

In [45]:
# Set the file directory for the processed files
output_dir = r"D:/Oceanographic_Kelp_Project/NOAA_all"

# List all CSV files for salt and temperature data
salt_files = [f for f in os.listdir(output_dir) if f.startswith("surf_salt_") and f.endswith(".csv")]
temp_files = [f for f in os.listdir(output_dir) if f.startswith("surf_temp_") and f.endswith(".csv")]

# Create empty lists to store DataFrames for salt and temperature
salt_dfs = []
temp_dfs = []

# Load and append each CSV for salt and temperature
for salt_file in salt_files:
    salt_df = pd.read_csv(os.path.join(output_dir, salt_file))
    salt_dfs.append(salt_df)
    print(f"Loaded salt data from {salt_file}")

for temp_file in temp_files:
    temp_df = pd.read_csv(os.path.join(output_dir, temp_file))
    temp_dfs.append(temp_df)
    print(f"Loaded temperature data from {temp_file}")

# Combine all salt DataFrames into one
combined_salt_df = pd.concat(salt_dfs, ignore_index=True)
combined_salt_df.to_csv(os.path.join(output_dir, "surf_salt_all.csv"), index=False)
print("Exported combined salt data to combined_surf_salt.csv")

# Combine all temperature DataFrames into one
combined_temp_df = pd.concat(temp_dfs, ignore_index=True)
combined_temp_df.to_csv(os.path.join(output_dir, "urf_temp_all.csv"), index=False)
print("Exported combined temperature data to combined_surf_temp.csv")


Loaded salt data from surf_salt_2008.csv
Loaded salt data from surf_salt_2009.csv
Loaded salt data from surf_salt_2010.csv
Loaded salt data from surf_salt_2011.csv
Loaded salt data from surf_salt_2012.csv
Loaded salt data from surf_salt_2013.csv
Loaded salt data from surf_salt_2014.csv
Loaded salt data from surf_salt_2015.csv
Loaded salt data from surf_salt_2016.csv
Loaded salt data from surf_salt_2017.csv
Loaded salt data from surf_salt_2018.csv
Loaded temperature data from surf_temp_2008.csv
Loaded temperature data from surf_temp_2009.csv
Loaded temperature data from surf_temp_2010.csv
Loaded temperature data from surf_temp_2011.csv
Loaded temperature data from surf_temp_2012.csv
Loaded temperature data from surf_temp_2013.csv
Loaded temperature data from surf_temp_2014.csv
Loaded temperature data from surf_temp_2015.csv
Loaded temperature data from surf_temp_2016.csv
Loaded temperature data from surf_temp_2017.csv
Loaded temperature data from surf_temp_2018.csv
Exported combined sal